# Build Clean Longitudinal Encounter Dataset

Purpose: create a clean, reusable patient encounter timeline that can later be merged with thread-level data to construct rolling pre-thread encounter exposure variables.

This notebook focuses on data preparation only. It does not calculate pre-thread covariates, merge with threads, process demographics/CCI, or run regression models.

The exported dataset is one row per encounter and retains the variables needed for downstream analyses such as total encounters in the previous 30 days, encounter-type-specific counts, unique providers, same-provider continuity, and department/provider continuity.

Design notes:

- `EmergencyVisits` is the primary ED source.
- `HospitalVisits` rows with `PT_TYPE=Emergency` are used only as ED fallback rows when absent from `EmergencyVisits`.
- Non-emergency `HospitalVisits` rows are kept as inpatient/outpatient/other hospital encounters.
- `VisitDetails` rows are kept as clinic/date-only encounters.
- Provider NPI is retained when available because it supports same-provider and continuity analyses.
- Department fields are retained when available.
- Diagnosis/reason context is harmonized only when already available from the visit sources.
- This notebook does not create `IS_ED` or `IS_ANY_VISIT`; downstream notebooks should use `visit_type_grouped`.
- No row-level patient or encounter examples are displayed.


## 1. Load And Clean Raw Encounter Data


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import re

import numpy as np
import pandas as pd

pd.options.display.max_columns = 160


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


def required_restricted_data_dir() -> Path:
    value = os.environ.get("JMIR_RESTRICTED_DATA_DIR")
    if not value:
        raise EnvironmentError(
            "Set JMIR_RESTRICTED_DATA_DIR to the authorized local data root before running. "
            "See data/README.md for the expected directory layout."
        )
    return Path(value).expanduser().resolve()


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
RUNS_DIR = OUTPUT_ROOT / "runs"
WORKING_OUTPUT_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()

RESTRICTED_DATA_DIR = required_restricted_data_dir()
SOURCE_RAW_DIR = Path(
    os.environ.get("JMIR_RAW_DATA_DIR", RESTRICTED_DATA_DIR / "raw")
).expanduser().resolve()
SOURCE_ADDED_DIR = Path(
    os.environ.get("JMIR_ADDITIONAL_DATA_DIR", SOURCE_RAW_DIR / "added October 25")
).expanduser().resolve()
SOURCE_CLEANED_DIR = Path(
    os.environ.get("JMIR_CLEANED_MESSAGE_DIR", RESTRICTED_DATA_DIR / "Cleaned")
).expanduser().resolve()

RUNS_DIR.mkdir(parents=True, exist_ok=True)
WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENCOUNTERS_CANONICAL_PATH = WORKING_OUTPUT_DIR / "encounters_unified_clean.parquet"
WRITE_CANONICAL_ENCOUNTERS = os.environ.get("JMIR_WRITE_CANONICAL_ENCOUNTERS", "1") == "1"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / f"clean_visit_encounters_{RUN_ID}"
if RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_DIR:", RUN_DIR)
print("SOURCE_RAW_DIR:", SOURCE_RAW_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_OUTPUT_DIR:", WORKING_OUTPUT_DIR)
print("ENCOUNTERS_CANONICAL_PATH:", ENCOUNTERS_CANONICAL_PATH)

### Helper Functions


In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.encode("ascii", "ignore")
        .str.decode("ascii")
        .str.strip()
        .str.replace(r"[^\w]+", "_", regex=True)
    )
    return out


def read_source_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(
        path,
        encoding="cp1252",
        encoding_errors="replace",
        sep=None,
        engine="python",
        on_bad_lines="skip",
    )
    return clean_columns(df)


def to_ts(s):
    out = pd.to_datetime(s, errors="coerce", utc=False)
    try:
        if getattr(out.dt, "tz", None) is not None:
            out = out.dt.tz_convert(None)
    except Exception:
        pass
    return out


def clean_id(s):
    return s.astype("string").str.strip()


def clean_text(s):
    return s.astype("string").str.strip().str.casefold()


def clean_npi(s):
    out = s.astype("string").str.replace(r"\D+", "", regex=True).str.strip()
    return out.mask(out.eq(""))


def dedup_exact(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    return df.loc[~df.duplicated(keys, keep="first")].copy()


def first_nonmissing(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA


def join_unique(s, max_items=20):
    vals = [str(x).strip() for x in s.dropna().unique() if str(x).strip()]
    vals = vals[:max_items]
    return "; ".join(vals) if vals else pd.NA


### Load Raw Visit Source Files


In [ ]:
source_paths = {
    "visit_details": SOURCE_RAW_DIR / "VisitDetails.txt",
    "visit_provider_details": SOURCE_RAW_DIR / "VisitProvider_Details.txt",
    "emergency_visits": SOURCE_ADDED_DIR / "EmergencyVisits.txt",
    "hospital_visits": SOURCE_ADDED_DIR / "HospitalVisits(Inpt,Out,Emergency)_updated.txt",
    "hospital_discharge_dx": SOURCE_ADDED_DIR / "HospitalVisits_DischargeDiagnosis_updated.txt",
    "clinical_visits_clinical_dx": SOURCE_ADDED_DIR / "ClinicalVisits_ClinicalDx_updated.txt",
    "clinical_visits_billed_dx": SOURCE_ADDED_DIR / "ClinicalVisits_BilledDx_updated.txt",
}

tables = {name: read_source_table(path) for name, path in source_paths.items()}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
    print("  columns:", list(df.columns))


### Build Provider Crosswalks


In [ ]:
VisitProvider_Details = tables["visit_provider_details"].copy()

vp = VisitProvider_Details.copy()
vp["PROV_NAME"] = vp["PROV_NAME"].astype("string").str.strip()
vp["PROV_TYPE"] = vp["PROV_TYPE"].astype("string").str.strip()
vp["SPECIALTY"] = vp["SPECIALTY"].astype("string").str.strip()
vp["NPI"] = clean_npi(vp["NPI"])
vp = vp.dropna(subset=["PROV_NAME"]).copy()

provider_by_name = (
    vp.groupby("PROV_NAME", as_index=False)
    .agg(
        provider_npi=("NPI", first_nonmissing),
        provider_type=("PROV_TYPE", first_nonmissing),
        provider_specialties=("SPECIALTY", join_unique),
    )
)

provider_by_npi = (
    vp.dropna(subset=["NPI"])
    .groupby("NPI", as_index=False)
    .agg(
        provider_name_from_npi=("PROV_NAME", first_nonmissing),
        provider_type_from_npi=("PROV_TYPE", first_nonmissing),
        provider_specialties_from_npi=("SPECIALTY", join_unique),
    )
)

print("provider_by_name rows:", len(provider_by_name))
print("provider_by_npi rows:", len(provider_by_npi))
print("Provider crosswalks built in memory for name/NPI enrichment.")


### Build Diagnosis Context References


In [ ]:
ClinicalDx = tables["clinical_visits_clinical_dx"].copy()
ClinicalBilled = tables["clinical_visits_billed_dx"].copy()
HospitalDischargeDx = tables["hospital_discharge_dx"].copy()


def prep_clinic_dx(df, code_col, name_col, code_out, name_out):
    out = df.copy()
    out["DMRN"] = clean_id(out["DMRN"])
    out["CONTACT_DATE"] = to_ts(out["CONTACT_DATE"]).dt.floor("D")
    out["ENC_TYPE"] = out["ENC_TYPE"].astype("string").str.strip()
    out["DEPARTMENT_NAME"] = out["DEPARTMENT_NAME"].astype("string").str.strip()
    out[code_col] = out[code_col].astype("string").str.strip()
    out[name_col] = out[name_col].astype("string").str.strip()
    out = out.dropna(subset=["DMRN", "CONTACT_DATE", "ENC_TYPE", "DEPARTMENT_NAME"])
    return (
        out.groupby(["DMRN", "CONTACT_DATE", "ENC_TYPE", "DEPARTMENT_NAME"], as_index=False)
        .agg(
            **{
                f"n_{code_out}": (code_col, lambda s: int(s.dropna().nunique())),
                code_out: (code_col, join_unique),
                name_out: (name_col, join_unique),
            }
        )
    )


clinic_clinical_dx = prep_clinic_dx(
    ClinicalDx, "clinical_Dx_Code", "CLINICAL_DX_NAME", "clinical_dx_codes", "clinical_dx_names"
)
clinic_billed_dx = prep_clinic_dx(
    ClinicalBilled, "Billing_Dx_code", "BILLING_DX_NAME", "billing_dx_codes", "billing_dx_names"
)

clinic_diagnosis_reference = clinic_clinical_dx.merge(
    clinic_billed_dx,
    on=["DMRN", "CONTACT_DATE", "ENC_TYPE", "DEPARTMENT_NAME"],
    how="outer",
)

hosp_dx = HospitalDischargeDx.copy()
hosp_dx["DMRN"] = clean_id(hosp_dx["DMRN"])
hosp_dx["DISCH_DATE_TIME"] = to_ts(hosp_dx["DISCH_DATE_TIME"])
hosp_dx["DISCH_DX_CODE"] = hosp_dx["DISCH_DX_CODE"].astype("string").str.strip()
hosp_dx["DISCH_DX_NAME"] = hosp_dx["DISCH_DX_NAME"].astype("string").str.strip()
hosp_discharge_dx_reference = (
    hosp_dx.dropna(subset=["DMRN", "DISCH_DATE_TIME"])
    .groupby(["DMRN", "DISCH_DATE_TIME"], as_index=False)
    .agg(
        n_discharge_dx_codes=("DISCH_DX_CODE", lambda s: int(s.dropna().nunique())),
        discharge_dx_codes=("DISCH_DX_CODE", join_unique),
        discharge_dx_names=("DISCH_DX_NAME", join_unique),
    )
)

print("clinic_diagnosis_reference rows:", len(clinic_diagnosis_reference))
print("hosp_discharge_dx_reference rows:", len(hosp_discharge_dx_reference))
print("Diagnosis context references built in memory for encounter enrichment.")
print("Hospital discharge diagnosis reference built in memory for encounter enrichment.")


## 2. Harmonize And Combine Encounters


In [ ]:
EmergencyVisits = tables["emergency_visits"].copy()

ev = EmergencyVisits.copy()
ev["DMRN"] = clean_id(ev["DMRN"])
ev["ADM_DATE_TIME"] = to_ts(ev["ADM_DATE_TIME"])
ev["DISCH_DATE_TIME"] = to_ts(ev["DISCH_DATE_TIME"])
ev["ATTENDING_PROV_NPI"] = clean_npi(ev["ATTENDING_PROV_NPI"])

rev_mask = ev["ADM_DATE_TIME"].notna() & ev["DISCH_DATE_TIME"].notna() & (ev["DISCH_DATE_TIME"] < ev["ADM_DATE_TIME"])
ev.loc[rev_mask, ["ADM_DATE_TIME", "DISCH_DATE_TIME"]] = ev.loc[rev_mask, ["DISCH_DATE_TIME", "ADM_DATE_TIME"]].to_numpy()
ev = ev.loc[ev["DMRN"].notna() & ev["ADM_DATE_TIME"].notna()].copy()

ev_h = (
    ev.groupby(["DMRN", "ADM_DATE_TIME"], as_index=False)
    .agg(
        DISCH_DATE_TIME=("DISCH_DATE_TIME", "max"),
        encounter_provider_npi=("ATTENDING_PROV_NPI", first_nonmissing),
        encounter_location_name=("DISCH_LOC", first_nonmissing),
        primary_dx_code=("PRIMARY_DISCH_DX_CODE", first_nonmissing),
        primary_dx_name=("PRIMARY_DISCH_DX_NAME", first_nonmissing),
        admit_source=("ADMIT_SOURCE", first_nonmissing),
        planned_admission=("PLANNED_ADMISSION", first_nonmissing),
        admission_type=("ADMISSION_TYPE", first_nonmissing),
        admit_dx_code=("ADMIT_DX_CODE", first_nonmissing),
        admit_dx_name=("ADMIT_DX_NAME", first_nonmissing),
        discharge_disposition=("Discharge_Disposition", first_nonmissing),
        financial_class=("FINANCIAL_CLASS", first_nonmissing),
        primary_insurance_payor=("PRIMARY_INSURANCE_PAYOR", first_nonmissing),
        chief_complaint=("CHIEF_COMPLAINT", join_unique),
    )
    .rename(columns={"ADM_DATE_TIME": "ENCOUNTER_START", "DISCH_DATE_TIME": "ENCOUNTER_END"})
)
ev_h["ENCOUNTER_END"] = ev_h["ENCOUNTER_END"].fillna(ev_h["ENCOUNTER_START"])
ev_h = ev_h.merge(provider_by_npi, left_on="encounter_provider_npi", right_on="NPI", how="left").drop(columns=["NPI"], errors="ignore")
ev_h["encounter_provider_name"] = ev_h["provider_name_from_npi"]
ev_h["provider_type"] = ev_h["provider_type_from_npi"]
ev_h["provider_specialties"] = ev_h["provider_specialties_from_npi"]
ev_h["encounter_department_name"] = pd.NA
ev_h["encounter_department_specialty"] = pd.NA
ev_h["SOURCE"] = "Emergency"
ev_h["SOURCE_DETAIL"] = "ED"
ev_h["TYPE_GROUP"] = "ED"

ev_h = ev_h.drop(columns=["provider_name_from_npi", "provider_type_from_npi", "provider_specialties_from_npi"], errors="ignore")
print("Emergency encounters:", len(ev_h))
print("Emergency encounters with provider name from NPI:", ev_h["encounter_provider_name"].notna().sum())
print("Emergency encounter rows built; row-level previews suppressed.")


### Build Hospital Encounters And Emergency Fallback


In [ ]:
HospitalVisits = tables["hospital_visits"].copy()

hv_all = HospitalVisits.copy()
hv_all["DMRN"] = clean_id(hv_all["DMRN"])
hv_all["ADM_DATE_TIME"] = to_ts(hv_all["ADM_DATE_TIME"])
hv_all["DISCH_DATE_TIME"] = to_ts(hv_all["DISCH_DATE_TIME"])
hv_all["_pt_type_norm"] = clean_text(hv_all["PT_TYPE"])
hv_all["ATTENDING_PROV_NAME"] = hv_all["ATTENDING_PROV_NAME"].astype("string").str.strip()
hv_all["ATTENDING_PROV_NPI"] = clean_npi(hv_all["ATTENDING_PROV_NPI"])

rev_mask = hv_all["ADM_DATE_TIME"].notna() & hv_all["DISCH_DATE_TIME"].notna() & (hv_all["DISCH_DATE_TIME"] < hv_all["ADM_DATE_TIME"])
hv_all.loc[rev_mask, ["ADM_DATE_TIME", "DISCH_DATE_TIME"]] = hv_all.loc[rev_mask, ["DISCH_DATE_TIME", "ADM_DATE_TIME"]].to_numpy()
hv_all = hv_all.loc[hv_all["DMRN"].notna() & hv_all["ADM_DATE_TIME"].notna()].copy()
hv_all["DISCH_DATE_TIME"] = hv_all["DISCH_DATE_TIME"].fillna(hv_all["ADM_DATE_TIME"])

def finalize_hospital_provider_fields(df):
    out = df.merge(provider_by_npi, left_on="encounter_provider_npi", right_on="NPI", how="left").drop(columns=["NPI"], errors="ignore")
    out["encounter_provider_name"] = out["encounter_provider_name"].fillna(out["provider_name_from_npi"])
    out["provider_type"] = out["provider_type_from_npi"]
    out["provider_specialties"] = out["provider_specialties_from_npi"]
    return out.drop(columns=["provider_name_from_npi", "provider_type_from_npi", "provider_specialties_from_npi"], errors="ignore")

hv_em = hv_all.loc[hv_all["_pt_type_norm"].eq("emergency")].copy()
hv_em_h = (
    hv_em.groupby(["DMRN", "ADM_DATE_TIME"], as_index=False)
    .agg(
        DISCH_DATE_TIME=("DISCH_DATE_TIME", "max"),
        encounter_provider_name=("ATTENDING_PROV_NAME", first_nonmissing),
        encounter_provider_npi=("ATTENDING_PROV_NPI", first_nonmissing),
        encounter_location_name=("DISCH_LOC", first_nonmissing),
        primary_dx_code=("PRIMARY_DISCH_DX_CODE", first_nonmissing),
        primary_dx_name=("PRIMARY_DISCH_DX_NAME", first_nonmissing),
        admit_source=("ADMIT_SOURCE", first_nonmissing),
        admission_type=("ADMISSION_TYPE", first_nonmissing),
        admit_dx_code=("ADMIT_DX_CODE", first_nonmissing),
        admit_dx_name=("ADMIT_DX_NAME", first_nonmissing),
        discharge_disposition=("Discharge_Disposition", first_nonmissing),
        financial_class=("FINANCIAL_CLASS", first_nonmissing),
        primary_insurance_payor=("PRIMARY_INSURANCE_PAYOR", first_nonmissing),
    )
    .rename(columns={"ADM_DATE_TIME": "ENCOUNTER_START", "DISCH_DATE_TIME": "ENCOUNTER_END"})
)
hv_em_h["ENCOUNTER_END"] = hv_em_h["ENCOUNTER_END"].fillna(hv_em_h["ENCOUNTER_START"])
hv_em_h = finalize_hospital_provider_fields(hv_em_h)
hv_em_h["encounter_department_name"] = pd.NA
hv_em_h["encounter_department_specialty"] = pd.NA

ev_h["_ed_key"] = ev_h["DMRN"].astype(str) + "|" + ev_h["ENCOUNTER_START"].dt.floor("min").astype(str)
hv_em_h["_ed_key"] = hv_em_h["DMRN"].astype(str) + "|" + hv_em_h["ENCOUNTER_START"].dt.floor("min").astype(str)
hv_em_fallback = hv_em_h.loc[~hv_em_h["_ed_key"].isin(set(ev_h["_ed_key"]))].copy()
ev_h = ev_h.drop(columns=["_ed_key"], errors="ignore")
hv_em_fallback = hv_em_fallback.drop(columns=["_ed_key"], errors="ignore")
hv_em_fallback["SOURCE"] = "Emergency"
hv_em_fallback["SOURCE_DETAIL"] = "ED_HospitalVisitsFallback"
hv_em_fallback["TYPE_GROUP"] = "ED"

hv = hv_all.loc[~hv_all["_pt_type_norm"].eq("emergency").fillna(False)].copy()
hv = dedup_exact(hv, keys=["DMRN", "ADM_DATE_TIME", "DISCH_DATE_TIME", "PT_TYPE"])
hv["SOURCE"] = "Hospital"
hv["SOURCE_DETAIL"] = hv["PT_TYPE"].astype("string").str.strip()
hv["TYPE_GROUP"] = np.select(
    [hv["_pt_type_norm"].eq("inpatient"), hv["_pt_type_norm"].eq("outpatient")],
    ["Hospital-Inpatient", "Hospital-Outpatient"],
    default="Hospital-Other",
)

hv_h = hv.rename(columns={"ADM_DATE_TIME": "ENCOUNTER_START", "DISCH_DATE_TIME": "ENCOUNTER_END"})[
    [
        "DMRN", "ENCOUNTER_START", "ENCOUNTER_END", "SOURCE", "SOURCE_DETAIL", "TYPE_GROUP",
        "ATTENDING_PROV_NAME", "ATTENDING_PROV_NPI", "DISCH_LOC", "PRIMARY_DISCH_DX_CODE", "PRIMARY_DISCH_DX_NAME",
        "ADMIT_SOURCE", "ADMISSION_TYPE", "ADMIT_DX_CODE", "ADMIT_DX_NAME", "Discharge_Disposition",
        "FINANCIAL_CLASS", "PRIMARY_INSURANCE_PAYOR",
    ]
].copy()
hv_h = hv_h.rename(columns={
    "ATTENDING_PROV_NAME": "encounter_provider_name",
    "ATTENDING_PROV_NPI": "encounter_provider_npi",
    "DISCH_LOC": "encounter_location_name",
    "PRIMARY_DISCH_DX_CODE": "primary_dx_code",
    "PRIMARY_DISCH_DX_NAME": "primary_dx_name",
    "ADMIT_SOURCE": "admit_source",
    "ADMISSION_TYPE": "admission_type",
    "ADMIT_DX_CODE": "admit_dx_code",
    "ADMIT_DX_NAME": "admit_dx_name",
    "Discharge_Disposition": "discharge_disposition",
    "FINANCIAL_CLASS": "financial_class",
    "PRIMARY_INSURANCE_PAYOR": "primary_insurance_payor",
})
hv_h = finalize_hospital_provider_fields(hv_h)
hv_h["encounter_department_name"] = pd.NA
hv_h["encounter_department_specialty"] = pd.NA

print("HV emergency fallback ED rows added:", len(hv_em_fallback))
print("Hospital non-ED encounters:", len(hv_h))
print("Hospital encounter rows built; row-level previews suppressed.")


### Build Clinic Encounters


In [ ]:
VisitDetails = tables["visit_details"].copy()

vd = VisitDetails.copy()
vd["DMRN"] = clean_id(vd["DMRN"])
vd["CONTACT_DATE"] = to_ts(vd["CONTACT_DATE"]).dt.floor("D")
vd["PROV_NAME"] = vd["PROV_NAME"].astype("string").str.strip()
vd["DEPARTMENT_NAME"] = vd["DEPARTMENT_NAME"].astype("string").str.strip()
vd["DEPT_SPECIALTY"] = vd["DEPT_SPECIALTY"].astype("string").str.strip()
vd["ENC_TYPE"] = vd["ENC_TYPE"].astype("string").str.strip()
vd["APPT_STATUS"] = vd["APPT_STATUS"].astype("string").str.strip()
vd["_enc_type_norm"] = clean_text(vd["ENC_TYPE"])
vd = vd.loc[vd["DMRN"].notna() & vd["CONTACT_DATE"].notna()].copy()

# Include provider and department in the clinic deduplication key so same-day visits to different clinicians/departments are preserved.
vd = dedup_exact(vd, keys=["DMRN", "CONTACT_DATE", "ENC_TYPE", "PROV_NAME", "DEPARTMENT_NAME"])

vd = vd.merge(provider_by_name, on="PROV_NAME", how="left")
vd = vd.merge(
    clinic_diagnosis_reference,
    on=["DMRN", "CONTACT_DATE", "ENC_TYPE", "DEPARTMENT_NAME"],
    how="left",
)

vd["SOURCE"] = "Clinic"
vd["SOURCE_DETAIL"] = vd["ENC_TYPE"]
vd["TYPE_GROUP"] = np.select(
    [vd["_enc_type_norm"].eq("office visit"), vd["_enc_type_norm"].eq("telemedicine")],
    ["Clinic-Office", "Clinic-Telemed"],
    default="Clinic-Other",
)

vd_h = vd.rename(columns={
    "CONTACT_DATE": "ENCOUNTER_START",
    "PROV_NAME": "encounter_provider_name",
    "provider_npi": "encounter_provider_npi",
    "DEPARTMENT_NAME": "encounter_department_name",
    "DEPT_SPECIALTY": "encounter_department_specialty",
})[
    [
        "DMRN", "ENCOUNTER_START", "SOURCE", "SOURCE_DETAIL", "TYPE_GROUP",
        "encounter_provider_name", "encounter_provider_npi", "provider_type", "provider_specialties",
        "encounter_department_name", "encounter_department_specialty", "VISIT_AGE", "APPT_STATUS",
        "n_clinical_dx_codes", "clinical_dx_codes", "clinical_dx_names",
        "n_billing_dx_codes", "billing_dx_codes", "billing_dx_names",
    ]
].copy()
vd_h["ENCOUNTER_END"] = vd_h["ENCOUNTER_START"]
vd_h["encounter_location_name"] = pd.NA
vd_h["primary_dx_code"] = pd.NA
vd_h["primary_dx_name"] = pd.NA

print("Clinic encounters:", len(vd_h))
print("Clinic encounters with provider name:", vd_h["encounter_provider_name"].notna().sum())
print("Clinic encounters with department name:", vd_h["encounter_department_name"].notna().sum())
print("Clinic encounter rows built; row-level previews suppressed.")


### Stack And Standardize Encounter Sources


In [ ]:
encounters_unified = pd.concat([ev_h, hv_em_fallback, hv_h, vd_h], ignore_index=True, sort=False)

encounters_unified["ENCOUNTER_START"] = to_ts(encounters_unified["ENCOUNTER_START"])
encounters_unified["ENCOUNTER_END"] = to_ts(encounters_unified["ENCOUNTER_END"])
encounters_unified["ENCOUNTER_END"] = encounters_unified["ENCOUNTER_END"].fillna(encounters_unified["ENCOUNTER_START"])


def map_visit_type_grouped(x):
    x = str(x)
    if x == "ED":
        return "ED"
    if x == "Hospital-Inpatient":
        return "Inpatient"
    if x == "Hospital-Outpatient":
        return "Hospital_Outpatient"
    if x.startswith("Clinic"):
        return "Clinic"
    return "Other"


encounters_unified["visit_type_detailed"] = encounters_unified["TYPE_GROUP"].astype(str)
encounters_unified["visit_type_grouped"] = encounters_unified["TYPE_GROUP"].apply(map_visit_type_grouped)

encounters_unified["ENCOUNTER_ID"] = (
    encounters_unified["DMRN"].astype(str) + "|" +
    encounters_unified["ENCOUNTER_START"].astype(str) + "|" +
    encounters_unified["SOURCE"].astype(str) + "|" +
    encounters_unified["TYPE_GROUP"].astype(str) + "|" +
    encounters_unified["encounter_provider_npi"].astype(str) + "|" +
    encounters_unified["encounter_provider_name"].astype(str) + "|" +
    encounters_unified["encounter_department_name"].astype(str)
)

assert encounters_unified["DMRN"].notna().all()
assert encounters_unified["ENCOUNTER_START"].notna().all()
assert encounters_unified["ENCOUNTER_END"].notna().all()
assert (encounters_unified["ENCOUNTER_END"] >= encounters_unified["ENCOUNTER_START"]).all()

print("encounters_unified rows:", len(encounters_unified))
print("patients:", encounters_unified["DMRN"].nunique())
print("rows with provider NPI:", encounters_unified["encounter_provider_npi"].notna().sum())
print("rows with provider name:", encounters_unified["encounter_provider_name"].notna().sum())
print("rows with department name:", encounters_unified["encounter_department_name"].notna().sum())


### Select Fields Needed For Encounter Timeline


In [ ]:
source_context_cols = [
    "primary_dx_code", "primary_dx_name",
    "admit_dx_code", "admit_dx_name", "chief_complaint",
    "clinical_dx_codes", "clinical_dx_names",
    "billing_dx_codes", "billing_dx_names",
]

encounter_base_cols = [
    "ENCOUNTER_ID", "DMRN", "ENCOUNTER_START", "ENCOUNTER_END",
    "SOURCE", "SOURCE_DETAIL", "TYPE_GROUP", "visit_type_detailed", "visit_type_grouped",
    "encounter_provider_npi", "encounter_provider_name", "provider_type", "provider_specialties",
    "encounter_department_name", "encounter_department_specialty",
]

encounter_cols = [c for c in encounter_base_cols + source_context_cols if c in encounters_unified.columns]
encounters_with_source_context = encounters_unified[encounter_cols].copy()

print("encounters_with_source_context rows:", len(encounters_with_source_context))
print("columns selected for harmonization:", len(encounter_cols))


### Harmonize Diagnosis And Reason Context


In [ ]:
def split_semicolon_values(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.upper() in {"<NA>", "NA", "NAN", "NONE"}:
        return []
    return [part.strip() for part in text.split(";") if part.strip()]


def combine_row_values(row, columns):
    out = []
    seen = set()
    for col in columns:
        if col not in row.index:
            continue
        for item in split_semicolon_values(row[col]):
            key = item.upper()
            if key not in seen:
                seen.add(key)
                out.append(item)
    return "; ".join(out) if out else pd.NA


def first_row_value(row, columns):
    for col in columns:
        if col not in row.index:
            continue
        values = split_semicolon_values(row[col])
        if values:
            return values[0]
    return pd.NA


dx_code_cols = ["admit_dx_code", "primary_dx_code", "clinical_dx_codes", "billing_dx_codes"]
dx_name_cols = ["admit_dx_name", "primary_dx_name", "clinical_dx_names", "billing_dx_names"]
reason_or_dx_text_cols = ["chief_complaint", "admit_dx_name", "primary_dx_name", "clinical_dx_names", "billing_dx_names"]

encounters_harmonized = encounters_with_source_context.copy()
encounters_harmonized["encounter_dx_codes_combined"] = encounters_harmonized.apply(lambda r: combine_row_values(r, dx_code_cols), axis=1)
encounters_harmonized["encounter_dx_names_combined"] = encounters_harmonized.apply(lambda r: combine_row_values(r, dx_name_cols), axis=1)
encounters_harmonized["encounter_reason_or_dx_text_combined"] = encounters_harmonized.apply(lambda r: combine_row_values(r, reason_or_dx_text_cols), axis=1)
encounters_harmonized["encounter_context_primary_text"] = encounters_harmonized.apply(lambda r: first_row_value(r, reason_or_dx_text_cols), axis=1)
encounters_harmonized["encounter_context_primary_dx_code"] = encounters_harmonized.apply(lambda r: first_row_value(r, dx_code_cols), axis=1)

final_encounter_cols = [
    "ENCOUNTER_ID", "DMRN", "ENCOUNTER_START", "ENCOUNTER_END",
    "SOURCE", "SOURCE_DETAIL", "TYPE_GROUP", "visit_type_detailed", "visit_type_grouped",
    "encounter_provider_npi", "encounter_provider_name", "provider_type", "provider_specialties",
    "encounter_department_name", "encounter_department_specialty",
    "encounter_dx_codes_combined", "encounter_dx_names_combined",
    "encounter_reason_or_dx_text_combined", "encounter_context_primary_text",
    "encounter_context_primary_dx_code",
]
final_encounter_cols = [c for c in final_encounter_cols if c in encounters_harmonized.columns]
encounters_final = encounters_harmonized[final_encounter_cols].copy()

print("encounters_final rows:", len(encounters_final))
print("final exported columns:", len(encounters_final.columns))


## 3. Quality Assurance


In [ ]:
encounters_per_patient = encounters_final.groupby("DMRN").size()
encounter_type_counts = (
    encounters_final["visit_type_grouped"]
    .value_counts(dropna=False)
    .rename_axis("visit_type_grouped")
    .reset_index(name="encounters")
)

qa_metrics = pd.DataFrame(
    [
        {"metric": "total_encounters", "value": int(len(encounters_final))},
        {"metric": "unique_patients", "value": int(encounters_final["DMRN"].nunique())},
        {"metric": "median_encounters_per_patient", "value": float(encounters_per_patient.median())},
        {"metric": "provider_npi_nonmissing_pct", "value": round(float(encounters_final["encounter_provider_npi"].notna().mean() * 100), 2)},
        {"metric": "provider_name_nonmissing_pct", "value": round(float(encounters_final["encounter_provider_name"].notna().mean() * 100), 2)},
        {"metric": "department_nonmissing_pct", "value": round(float(encounters_final["encounter_department_name"].notna().mean() * 100), 2)},
    ]
)

print("Concise encounter construction QA")
display(qa_metrics)
display(encounter_type_counts)


## 4. Export Final Encounter Dataset


In [ ]:
encounters_path = RUN_DIR / "encounters_unified_clean.parquet"
qa_summary_path = RUN_DIR / "qa_encounters_summary.json"

encounters_final.to_parquet(encounters_path, index=False)

canonical_encounters_path = None
if WRITE_CANONICAL_ENCOUNTERS:
    WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    canonical_encounters_path = ENCOUNTERS_CANONICAL_PATH
    encounters_final.to_parquet(canonical_encounters_path, index=False)

qa_summary = {
    "run_id": RUN_ID,
    "source_raw_dir": str(SOURCE_RAW_DIR),
    "hospital_emergency_fallback_ed_rows_added": int(len(hv_em_fallback)),
    "encounter_rows": int(len(encounters_final)),
    "encounter_patients": int(encounters_final["DMRN"].nunique()),
    "median_encounters_per_patient": float(encounters_per_patient.median()),
    "rows_with_provider_npi": int(encounters_final["encounter_provider_npi"].notna().sum()),
    "rows_with_provider_name": int(encounters_final["encounter_provider_name"].notna().sum()),
    "rows_with_department_name": int(encounters_final["encounter_department_name"].notna().sum()),
    "provider_npi_nonmissing_pct": round(float(encounters_final["encounter_provider_npi"].notna().mean() * 100), 2),
    "provider_name_nonmissing_pct": round(float(encounters_final["encounter_provider_name"].notna().mean() * 100), 2),
    "department_nonmissing_pct": round(float(encounters_final["encounter_department_name"].notna().mean() * 100), 2),
    "encounter_type_counts": encounter_type_counts.to_dict(orient="records"),
    "diagnosis_context_counts": {
        "rows_with_combined_dx_codes": int(encounters_final["encounter_dx_codes_combined"].notna().sum()),
        "rows_with_combined_dx_names": int(encounters_final["encounter_dx_names_combined"].notna().sum()),
        "rows_with_reason_or_dx_text": int(encounters_final["encounter_reason_or_dx_text_combined"].notna().sum()),
    },
    "output_files": {
        "encounters_unified_clean_run_copy": str(encounters_path),
        "encounters_unified_clean_canonical": str(canonical_encounters_path) if canonical_encounters_path is not None else None,
        "qa_encounters_summary": str(qa_summary_path),
    },
    "exported_columns": list(encounters_final.columns),
}

with open(qa_summary_path, "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2, default=str)

print("Saved encounter handoff outputs:")
for path in [encounters_path, canonical_encounters_path, qa_summary_path]:
    if path is not None:
        print(" -", path)


## Expected Main Output

Every run writes to a unique folder under `runs/clean_visit_encounters_YYYYMMDD_HHMMSS/`.

Working outputs:

- `encounters_unified_clean.parquet`: one row per encounter, containing patient ID, encounter ID, encounter start/end time, encounter type, provider NPI/name/type, department, and harmonized diagnosis/reason context fields.
- `qa_encounters_summary.json`: concise aggregate QA counts for total encounters, unique patients, encounter types, median encounters per patient, provider completeness, and department completeness.

A stable working copy is also written to `outputs/working_data/encounters_unified_clean.parquet` by default. This is the encounter handoff file for downstream rolling pre-thread encounter covariate construction.

Temporary variables used for deduplication or harmonization are not exported.
